# Lógica difusa: cuando «verdadero» y «falso» no bastan

**Facsímil 12 · Sistemas expertos y razonamiento** — capítulo 3 (lógica difusa y factores de certeza).

La lógica clásica solo conoce dos respuestas: sí o no, 1 o 0. Pero el mundo está lleno de matices.
¿A partir de qué temperatura exacta el agua está «caliente»? ¿A los 40 grados sí y a los 39,9 no?
Eso no encaja con cómo razonamos las personas. La **lógica difusa**, propuesta por Lotfi Zadeh en
1965, sustituye esa frontera tajante por una **pertenencia graduada**: algo puede pertenecer a un
conjunto «un 0,7», ni del todo ni nada. En este cuaderno construyes, en Python puro y a mano, todas
las piezas: funciones de pertenencia, operadores difusos, factores de certeza al estilo del sistema
experto MYCIN y un **controlador de Mamdani completo** que decide la propina de un restaurante a
partir del servicio y la comida. Todo de principio a fin, con la salida concreta y su superficie de
control.

> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

## 1. de la frontera tajante al grado de pertenencia

En la teoría de conjuntos clásica, un elemento **está o no está** en un conjunto: la pertenencia es
una función que solo vale 0 o 1. Zadeh propuso relajar esa regla. En un **conjunto difuso**, la
pertenencia es un número real en el intervalo `[0, 1]`: 0 es «no pertenece», 1 es «pertenece del
todo» y los valores intermedios miden *cuánto*.

La herramienta que asigna ese grado se llama **función de pertenencia**. Las tres formas más usadas
son la **triangular**, la **trapezoidal** y la **gaussiana**. Vamos a implementarlas con numpy: nada
de librerías mágicas. En producción se usa `scikit-fuzzy`, pero aquí lo hacemos a mano para que veas
que por dentro no hay misterio, solo aritmética.

In [ ]:
# En Colab ya viene todo. En tu maquina:  pip install numpy matplotlib
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def trimf(x, a, b, c):
    "Funcion triangular: sube de a a b, baja de b a c."
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    izq = (x > a) & (x <= b)
    der = (x > b) & (x < c)
    if b > a:
        y[izq] = (x[izq] - a) / (b - a)
    if c > b:
        y[der] = (c - x[der]) / (c - b)
    y[x == b] = 1.0
    return y

def trapmf(x, a, b, c, d):
    "Funcion trapezoidal: sube a->b, plana b->c, baja c->d."
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    sube = (x > a) & (x < b)
    plano = (x >= b) & (x <= c)
    baja = (x > c) & (x < d)
    if b > a:
        y[sube] = (x[sube] - a) / (b - a)
    y[plano] = 1.0
    if d > c:
        y[baja] = (d - x[baja]) / (d - c)
    return y

def gaussmf(x, mu, sigma):
    "Funcion gaussiana: campana centrada en mu, anchura sigma."
    x = np.asarray(x, dtype=float)
    return np.exp(-((x - mu) ** 2) / (2.0 * sigma ** 2))

print("Funciones de pertenencia listas.")

## 2. las tres formas básicas, dibujadas

Cada función describe la misma idea —«cuánto pertenece `x` a esta etiqueta»— con un perfil distinto.
La triangular es la más simple (un pico), la trapezoidal añade una meseta de pertenencia plena, y la
gaussiana da una transición suave sin esquinas. Dibujémoslas sobre un mismo eje para comparar.

In [ ]:
x = np.linspace(0, 10, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, trimf(x, 2, 5, 8), label="triangular  trimf(2, 5, 8)")
ax.plot(x, trapmf(x, 1, 3, 7, 9), label="trapezoidal trapmf(1, 3, 7, 9)")
ax.plot(x, gaussmf(x, 5, 1.5), label="gaussiana   gaussmf(mu=5, sigma=1.5)")
ax.set_xlabel("x"); ax.set_ylabel("grado de pertenencia")
ax.set_title("Tres funciones de pertenencia"); ax.set_ylim(-0.05, 1.1)
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 3. etiquetas que se solapan: la temperatura

Aquí está la idea central. Una misma variable —la temperatura— se describe con varias etiquetas
difusas que **se solapan**: «frío», «templado» y «caliente». A 26 grados no eres «templado» del todo
ni «caliente» del todo: eres un poco de cada cosa. Ese solapamiento es lo que permite una transición
suave, sin saltos bruscos cuando cruzas un valor concreto.

Definimos «frío» como una trapezoidal (pleno hasta los 12 grados, desaparece a los 18), «templado»
como una gaussiana centrada en 22, y «caliente» como una trapezoidal que arranca a los 24.

In [ ]:
def frio(t):     return trapmf(t, 0, 0, 12, 18)
def templado(t): return gaussmf(t, 22, 4)
def caliente(t): return trapmf(t, 24, 30, 40, 40)

t = np.linspace(0, 40, 500)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, frio(t),     label="frio")
ax.plot(t, templado(t), label="templado")
ax.plot(t, caliente(t), label="caliente")
ax.axvline(26, color="gray", ls="--", lw=1)
ax.set_xlabel("temperatura (grados C)"); ax.set_ylabel("pertenencia")
ax.set_title("Etiquetas difusas solapadas sobre la temperatura")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
temp = 26.0
print(f"A {temp} grados C, la temperatura pertenece a:")
print(f"  frio     = {frio([temp])[0]:.3f}")
print(f"  templado = {templado([temp])[0]:.3f}")
print(f"  caliente = {caliente([temp])[0]:.3f}")

A 26 grados el resultado es **frío = 0,000**, **templado = 0,607** y **caliente = 0,333**. La lectura
es directa: a 26 grados estamos sobre todo en zona templada (0,607), pero ya asoma lo caliente
(0,333). No hay contradicción en pertenecer a dos conjuntos a la vez; precisamente esa es la gracia.
Fíjate en que los grados **no tienen por qué sumar 1**: no son probabilidades, son grados de
compatibilidad independientes con cada etiqueta.

## 4. operadores difusos: Y, O, NO

Con valores entre 0 y 1 necesitamos redefinir las operaciones lógicas. La elección clásica de Zadeh
es la más sencilla y la que usaremos:

- **Y (conjunción)** → el **mínimo**: `a Y b = min(a, b)`. La cadena es tan fuerte como su eslabón
  más débil.
- **O (disyunción)** → el **máximo**: `a O b = max(a, b)`. Basta con que se cumpla una de las partes.
- **NO (negación)** → el **complemento**: `NO a = 1 - a`.

Estas reglas generalizan la lógica booleana: si solo metes ceros y unos, se comportan exactamente
igual que el `and`, el `or` y el `not` de toda la vida. (El mínimo y el máximo son la opción estándar,
pero existen otras familias, las llamadas t-normas y t-conormas, como el producto `a*b` para la Y.)

In [ ]:
def f_y(a, b):  return min(a, b)
def f_o(a, b):  return max(a, b)
def f_no(a):    return 1 - a

a, b = 0.7, 0.4
print(f"a = {a},  b = {b}")
print(f"  a Y b  = min(a, b) = {f_y(a, b)}")
print(f"  a O b  = max(a, b) = {f_o(a, b)}")
print(f"  NO a   = 1 - a     = {f_no(a):.1f}")

Con `a = 0,7` y `b = 0,4` obtenemos **a Y b = 0,4** (el mínimo, manda el más débil), **a O b = 0,7**
(el máximo) y **NO a = 0,3**. Tres líneas de aritmética que sustituyen a toda una tabla de verdad.

## 5. factores de certeza al estilo MYCIN

La lógica difusa modela la **vaguedad** («¿cuánto de caliente?»). Los **factores de certeza** (CF)
modelan algo distinto: la **confianza** en una afirmación. Nacieron en MYCIN, un sistema experto de
los años 70 para diagnosticar infecciones, donde las reglas no eran tajantes sino del tipo «*si* el
cultivo es positivo, *entonces* es tal bacteria, con certeza 0,7».

Un CF es un número en `[-1, 1]`: **+1** es certeza total a favor, **-1** certeza total en contra, y
**0** es «ni idea». Lo interesante es cómo se **combinan** dos evidencias sobre la misma conclusión.
MYCIN usa estas fórmulas, según los signos de las dos certezas `CF1` y `CF2`:

- Ambas a favor (`>= 0`):    `CF = CF1 + CF2 · (1 - CF1)`
- Ambas en contra (`<= 0`):  `CF = CF1 + CF2 · (1 + CF1)`
- Signos opuestos:           `CF = (CF1 + CF2) / (1 - min(|CF1|, |CF2|))`

Y para una **regla con antecedente incierto**, la certeza de la conclusión es la de la regla
multiplicada por la del antecedente (recortada a 0 si el antecedente es negativo):
`CF_conclusión = CF_regla · max(0, CF_antecedente)`.

In [ ]:
def cf_combina(c1, c2):
    "Combina dos factores de certeza sobre la misma conclusion (MYCIN)."
    if c1 >= 0 and c2 >= 0:
        return c1 + c2 * (1 - c1)
    if c1 <= 0 and c2 <= 0:
        return c1 + c2 * (1 + c1)
    return (c1 + c2) / (1 - min(abs(c1), abs(c2)))

print("Dos evidencias A FAVOR  CF1=0.6, CF2=0.4 :", round(cf_combina(0.6, 0.4), 4))
print("Evidencias en CONFLICTO CF1=0.6, CF2=-0.3:", round(cf_combina(0.6, -0.3), 4))
print("Dos evidencias EN CONTRA CF1=-0.5, CF2=-0.4:", round(cf_combina(-0.5, -0.4), 4))

cf_regla, cf_antecedente = 0.8, 0.7
cf_conclusion = cf_regla * max(0, cf_antecedente)
print(f"Regla CF=0.8 con antecedente CF=0.7 -> conclusion CF = {cf_conclusion:.2f}")

Lee con calma los cuatro resultados:

- **Dos evidencias a favor (0,6 y 0,4) → 0,76.** Se refuerzan: la confianza sube por encima de
  cualquiera de las dos, pero **sin pasarse de 1**. Cada nueva prueba a favor aporta cada vez menos.
- **Conflicto (0,6 y -0,3) → 0,43.** Una evidencia a favor y otra en contra se restan parcialmente.
- **Dos en contra (-0,5 y -0,4) → -0,70.** Simétrico al primer caso, pero hacia la desconfianza.
- **Regla 0,8 con antecedente 0,7 → 0,56.** Si no estás seguro del «si», tampoco puedes estarlo del
  «entonces»: la certeza se atenúa (0,8 × 0,7).

Los CF fueron un apaño ingenioso anterior a que las redes bayesianas dieran un marco probabilístico
riguroso; aun así ilustran muy bien el problema de **acumular evidencia incierta**.

## 6. inferencia de Mamdani: un controlador completo

Llegamos al plato fuerte. Vamos a construir un **controlador difuso de Mamdani** de principio a fin.
El ejemplo clásico: decidir la **propina** (entre 0 y 25 %) de un restaurante a partir de dos
entradas, la **calidad del servicio** y la **calidad de la comida** (ambas de 0 a 10).

La inferencia de Mamdani tiene cuatro pasos:

1. **Fuzzificar**: convertir las entradas nítidas (servicio = 6,5) en grados de pertenencia a cada
   etiqueta (aceptable = 0,7, excelente = 0,3...).
2. **Evaluar las reglas** SI-ENTONCES: cada regla produce una fuerza de activación.
3. **Agregar**: recortar cada conjunto de salida a la fuerza de su regla y unirlos todos en una sola
   forma difusa.
4. **Defuzzificar**: comprimir esa forma en un único número. Usaremos el **centroide** (el «centro de
   masas» del área), calculado numéricamente con numpy.

Empezamos definiendo los conjuntos de entrada y de salida.

In [ ]:
P = np.linspace(0, 25, 1001)   # universo de la propina, en %

# conjuntos de entrada (servicio y comida, de 0 a 10)
def serv_malo(x):  return trimf(x, 0, 0, 5)
def serv_acep(x):  return trimf(x, 0, 5, 10)
def serv_exc(x):   return trimf(x, 5, 10, 10)
def com_mala(x):   return trimf(x, 0, 0, 5)
def com_buena(x):  return trimf(x, 5, 10, 10)

# conjuntos de salida (propina)
def prop_baja(x):  return trimf(x, 0, 0, 13)
def prop_media(x): return trimf(x, 0, 13, 25)
def prop_alta(x):  return trimf(x, 13, 25, 25)

xs = np.linspace(0, 10, 400)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(xs, serv_malo(xs), label="malo")
ax[0].plot(xs, serv_acep(xs), label="aceptable")
ax[0].plot(xs, serv_exc(xs),  label="excelente")
ax[0].set_title("servicio"); ax[0].set_xlabel("0-10"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(P, prop_baja(P),  label="baja")
ax[1].plot(P, prop_media(P), label="media")
ax[1].plot(P, prop_alta(P),  label="alta")
ax[1].set_title("propina"); ax[1].set_xlabel("%"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. las reglas SI-ENTONCES

El conocimiento del experto se escribe como reglas en lenguaje casi natural. Para la propina bastan
tres:

1. **SI** el servicio es malo **O** la comida es mala **ENTONCES** la propina es baja.
2. **SI** el servicio es aceptable **ENTONCES** la propina es media.
3. **SI** el servicio es excelente **O** la comida es buena **ENTONCES** la propina es alta.

El «O» se traduce con `max` (lo vimos antes). La fuerza de cada regla es el grado con que se cumple su
antecedente. Implementamos la inferencia completa en una sola función.

In [ ]:
def mamdani(serv, com, verbose=False):
    # 1) fuzzificacion
    m_malo  = serv_malo([serv])[0]
    m_acep  = serv_acep([serv])[0]
    m_exc   = serv_exc([serv])[0]
    f_mala  = com_mala([com])[0]
    f_buena = com_buena([com])[0]
    # 2) evaluacion de reglas (el "O" es max)
    r1 = max(m_malo, f_mala)   # -> propina baja
    r2 = m_acep                # -> propina media
    r3 = max(m_exc, f_buena)   # -> propina alta
    # 3) agregacion: recortar cada salida (min) y unir (max)
    agg = np.maximum.reduce([
        np.minimum(r1, prop_baja(P)),
        np.minimum(r2, prop_media(P)),
        np.minimum(r3, prop_alta(P)),
    ])
    # 4) defuzzificacion por centroide
    centro = np.sum(P * agg) / np.sum(agg) if agg.sum() > 0 else 0.0
    if verbose:
        print(f"entradas: servicio={serv}, comida={com}")
        print(f"  fuzzificacion servicio: malo={m_malo:.3f} aceptable={m_acep:.3f} excelente={m_exc:.3f}")
        print(f"  fuzzificacion comida:   mala={f_mala:.3f} buena={f_buena:.3f}")
        print(f"  reglas -> R1 baja={r1:.3f}  R2 media={r2:.3f}  R3 alta={r3:.3f}")
        print(f"  PROPINA (centroide) = {centro:.4f} %")
    return centro, agg

centro, agg = mamdani(6.5, 9.8, verbose=True)

Sigamos el caso **servicio = 6,5, comida = 9,8** (buen servicio, comida excelente):

- **Fuzzificación.** Un 6,5 de servicio es `malo = 0,000`, `aceptable = 0,700` y `excelente = 0,300`.
  Un 9,8 de comida es `buena = 0,960`.
- **Reglas.** R1 (baja) = 0,000, R2 (media) = 0,700, R3 (alta) = max(0,300, 0,960) = **0,960**.
- **Defuzzificación.** El centroide del área agregada cae en **14,81 %**.

Tira sobre todo la regla de la propina alta (0,960 por la comida excelente), pero la propina media
(0,700) la frena un poco: el resultado, 14,81 %, queda entre la zona media y la alta. Eso es razonar
con grados.

## 8. ver la defuzzificación: el área y su centro de masas

La forma azul es el conjunto difuso agregado (la unión de las tres salidas recortadas). El centroide
es el valor `x` que la parte en dos mitades de igual «peso»: la fórmula es
`centroide = Σ(x · μ(x)) / Σ(μ(x))`, una media de las posiciones ponderada por la pertenencia. La
línea roja marca dónde cae.

In [ ]:
centro, agg = mamdani(6.5, 9.8)
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(P, agg, alpha=0.4, label="conjunto agregado")
ax.plot(P, agg, lw=1.5)
ax.axvline(centro, color="red", lw=2, label=f"centroide = {centro:.2f} %")
ax.set_xlabel("propina (%)"); ax.set_ylabel("pertenencia")
ax.set_title("Agregacion de las reglas y defuzzificacion por centroide")
ax.legend(); ax.grid(alpha=0.3)
plt.show()
print(f"Propina recomendada para (servicio=6.5, comida=9.8): {centro:.2f} %")

## 9. la superficie de control

Un controlador no es una respuesta suelta, sino una **función** que asigna una salida a cada
combinación de entradas. Si barremos todos los valores de servicio y comida y dibujamos la propina
resultante como un mapa de calor, obtenemos la **superficie de control**: la huella completa del
sistema. Fíjate en lo suave que es; no hay escalones.

In [ ]:
n = 41
serv_grid = np.linspace(0, 10, n)
com_grid = np.linspace(0, 10, n)
Z = np.zeros((n, n))
for i, sv in enumerate(serv_grid):
    for j, cm in enumerate(com_grid):
        Z[i, j] = mamdani(sv, cm)[0]

fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(Z, origin="lower", extent=[0, 10, 0, 10], aspect="auto", cmap="viridis")
ax.set_xlabel("comida"); ax.set_ylabel("servicio")
ax.set_title("Superficie de control: propina (%)")
fig.colorbar(im, ax=ax, label="propina (%)")
plt.show()
print(f"propina minima en la malla: {Z.min():.2f} %   maxima: {Z.max():.2f} %")

## 10. la difusa frente al controlador «de toda la vida»

¿Para qué tanto lío? Comparémoslo con un controlador clásico **por umbrales**: calcula la media de
servicio y comida y devuelve una propina fija según en qué tramo cae (baja, media o alta). Es lo que
saldría de un `if/elif/else`. Vamos a barrer el caso en que servicio y comida valen lo mismo y a
superponer las dos curvas.

In [ ]:
def umbral(serv, com):
    m = (serv + com) / 2.0
    if m < 3.34:   return 5.0
    elif m < 6.67: return 15.0
    else:          return 22.0

vals = np.linspace(0, 10, 200)
y_difusa = np.array([mamdani(v, v)[0] for v in vals])
y_umbral = np.array([umbral(v, v) for v in vals])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(vals, y_umbral, label="controlador por umbrales", lw=2, drawstyle="steps-post")
ax.plot(vals, y_difusa, label="controlador difuso (Mamdani)", lw=2)
ax.set_xlabel("servicio = comida"); ax.set_ylabel("propina (%)")
ax.set_title("Transicion brusca (umbrales) frente a suave (difusa)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

for v in [6.6, 6.8]:
    print(f"valor={v}: umbrales={umbral(v, v):.1f} %   difusa={mamdani(v, v)[0]:.2f} %")

El contraste es claro. El controlador por umbrales **salta**: justo al cruzar 6,67 pasa de
**15,0 % a 22,0 %** de golpe. Dos comidas casi idénticas (un 6,6 y un 6,8) reciben propinas que se
diferencian en 7 puntos enteros solo por estar a un lado u otro de la raya. El controlador difuso,
en cambio, pasa de **13,22 % a 13,37 %**: sube de forma **continua y proporcional**. Esa suavidad es
la razón por la que la lógica difusa triunfó en el control de electrodomésticos (lavadoras, aires
acondicionados, cámaras): evita los tirones que delatan a un sistema de reglas rígidas.

## 11. pruébalo tú

Cambia los valores de `mi_servicio` y `mi_comida` (entre 0 y 10) y vuelve a ejecutar. Ideas para
experimentar:

- Pon servicio y comida muy bajos (1 y 2): verás cómo se dispara la regla de la propina baja.
- Pon un servicio mediocre (5) y una comida excelente (10): observa cómo compiten las reglas.
- Modifica los rangos de los conjuntos (por ejemplo, dónde acaba «aceptable») y mira cómo se deforma
  la superficie de control.

In [ ]:
mi_servicio = 3.0
mi_comida = 8.5
centro, _ = mamdani(mi_servicio, mi_comida, verbose=True)
print(f"\n>> Propina recomendada: {centro:.2f} %")

## 12. errores comunes

- **Confundir difuso con probabilístico.** Un grado de pertenencia de 0,607 no es «un 60,7 % de
  probabilidad de que sea templado»; es «es templado en un grado 0,607». Por eso los grados no suman
  1. Vaguedad e incertidumbre son cosas distintas.
- **Olvidar que las etiquetas deben solaparse.** Si los conjuntos no se pisan, reaparecen los saltos
  bruscos que queríamos evitar: vuelves a tener un `if/else` disfrazado.
- **Elegir mal la defuzzificación.** El centroide es el método más común, pero hay otros (máximo,
  media de máximos...) y pueden dar resultados distintos para la misma forma agregada. No es un
  detalle menor.
- **Creer que la difusa «aprende».** No hay entrenamiento: las reglas y los conjuntos los pones tú
  (o el experto). Su fuerza es la interpretabilidad, no el ajuste automático a datos.
- **Reinventar la rueda en producción.** Aquí lo hicimos a mano para entenderlo; en un proyecto real
  usarías `scikit-fuzzy`, que trae las funciones de pertenencia, el motor de reglas y varios métodos
  de defuzzificación ya probados.

## 13. qué te llevas

- Un **conjunto difuso** sustituye la pertenencia 0/1 por un grado en `[0, 1]`, definido por una
  **función de pertenencia** (triangular, trapezoidal o gaussiana).
- Los **operadores** se redefinen: **Y = min**, **O = max**, **NO = 1 − x**. Generalizan la lógica
  booleana sin romperla.
- Los **factores de certeza** de MYCIN combinan evidencia incierta con fórmulas según el signo; dos
  pruebas a favor (0,6 y 0,4) dan 0,76, y un antecedente dudoso atenúa la conclusión (0,8 × 0,7 = 0,56).
- La **inferencia de Mamdani** encadena fuzzificar → reglas → agregar → defuzzificar. Para
  servicio = 6,5 y comida = 9,8, nuestro controlador recomienda **14,81 %** de propina.
- La **superficie de control** muestra que la salida cambia de forma **suave**, frente a los saltos de
  un controlador por umbrales (de 15 % a 22 % de golpe). Esa continuidad es la ventaja práctica de la
  lógica difusa.

En el próximo paso del facsímil verás cómo estas ideas conectan con los sistemas expertos completos y
con el razonamiento bajo incertidumbre que más adelante formalizarán las redes bayesianas.

---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*